# News Deserts in Region Syddanmark — Reproducing the report

This notebook reproduces the numbers in `news_deserts_corrected.tex` (report sections 4–4.5).

**Input files:**
- `news_desert_coding_template_prefilled.xlsx` — completed newspaper-municipality mapping
  (60 rows, 34 unique newspapers, 22 covered municipalities)
- `dk_municipalities.csv` — all 98 municipalities with population figures
- `kommuner.geojson` — municipal boundary polygons from Dataforsyningen
- `socioeconomic_data.csv` — mean income (INDKP101), higher-education share (HFUDD10), voter turnout KV21

**Study area:** 23 municipalities in Region Syddanmark. Horsens (code 615) is tagged as Syddanmark in `dk_municipalities.csv` because it falls within JFM's coverage footprint, even though it administratively belongs to Region Midtjylland.

**Main differences from `analyse.ipynb`:**
1. Reads the *completed* coding template (not the empty 5-row version)
2. Filters to the 23 study municipalities (not all 98)
3. Uses both Queen *and* KNN-4 weights — KNN-4 computed from projected centroids (EPSG:25832)
4. Reports Spearman alongside Pearson

## 0. Imports and configuration

Loads all required libraries and sets file paths. All data files should be in the same folder as this notebook, except `kommuner.geojson` which is hosted separately at Dataforsyningen (114 MB — see README for download link).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from libpysal.weights import Queen, KNN
from esda.moran import Moran, Moran_Local
from scipy.stats import spearmanr, pearsonr

# File paths — all input files should be in the same folder as this notebook
GEOJSON_PATH = "kommuner.geojson"
CODING_XLSX  = "news_desert_coding_template_prefilled.xlsx"   # the completed coding template
MUNI_CSV     = "dk_municipalities.csv"
SOCIO_CSV    = "socioeconomic_data.csv"

# Random seed for reproducibility of permutation-based p-values
RANDOM_SEED = 42

%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'sans-serif'
print("✓ Imports klar")

## 1. Load municipality data (all 98)

`dk_municipalities.csv` contains name, region, and population for each municipality. Note that Horsens (615) is tagged `region = "Syddanmark"` here because the analysis covers JFM's coverage footprint, which extends into Horsens.

In [ ]:
muni = pd.read_csv(MUNI_CSV).rename(columns={"population_approx": "population"})
print(f"Antal kommuner: {len(muni)}")
print(f"Samlet befolkning: {muni['population'].sum():,}")
print(muni.groupby('region')['population'].agg(['count', 'sum']).rename(
    columns={'count': 'kommuner', 'sum': 'befolkning'}))

## 2. Load newspaper data from the completed coding template

Expected counts (report §4.1):
- 60 newspaper-to-municipality mapping rows
- 34 unique newspapers
- 22 covered municipalities within the study area

In [ ]:
raw = pd.read_excel(CODING_XLSX, sheet_name="newspaper_municipality_map")
raw = raw.dropna(subset=["kommune_kode", "newspaper_id"])
raw["kommune_kode"] = raw["kommune_kode"].astype(int)

print(f"Mapping-rækker:    {len(raw)}")
print(f"Unikke aviser:     {raw['newspaper_id'].nunique()}")
print(f"Dækkede kommuner:  {raw['kommune_kode'].nunique()}")

# Publisher concentration (report §4.1)
pub_counts = raw['publisher'].value_counts()
print("\nPublisher-fordeling (rækker):")
print(pub_counts)

# Publisher string in the completed template is "JFM" (not "Jysk Fynske Medier")
JFM_LABEL = "JFM"
jfm_titles = raw.loc[raw['publisher']==JFM_LABEL, 'newspaper_id'].nunique()
total_titles = raw['newspaper_id'].nunique()
print(f"\nJFM andel af titler: {jfm_titles}/{total_titles} = {jfm_titles/total_titles:.1%}")
print(f"JFM andel af rækker: {pub_counts.get(JFM_LABEL,0)}/{len(raw)} = "
      f"{pub_counts.get(JFM_LABEL,0)/len(raw):.1%}")

## 3. Aggregate to municipality level

Count unique newspapers per municipality, plus a breakdown by publication frequency and payment type (free vs. paid).

In [ ]:
def nunique_where(s_col, mask_col, val):
    """Antal unikke avis-id'er hvor mask_col == val."""
    def f(idx):
        sub = raw.loc[idx.index]
        return sub.loc[sub[mask_col] == val, s_col].nunique()
    return f

agg = (raw.groupby("kommune_kode")
          .agg(newspaper_count=("newspaper_id", "nunique"),
               weekly_count =("newspaper_id", nunique_where("newspaper_id","frequency","weekly")),
               daily_count  =("newspaper_id", nunique_where("newspaper_id","frequency","daily")),
               paid_count   =("newspaper_id", nunique_where("newspaper_id","type","paid")),
               free_count   =("newspaper_id", nunique_where("newspaper_id","type","free")))
          .reset_index())
print(f"{len(agg)} kommuner med ≥1 mapped avis")
agg.head()

## 4. Per-capita score and filtering to the 23-municipality study area

`score_per_10k = newspaper_count / population × 10,000`

Expected values (report §4.2):
- mean 0.621, median 0.573
- range 0.000 (Fanø) to 1.734 (Ærø)
- 1 strict news desert: Fanø

In [ ]:
scores = muni.merge(agg, on="kommune_kode", how="left")
for c in ["newspaper_count","weekly_count","daily_count","paid_count","free_count"]:
    scores[c] = scores[c].fillna(0).astype(int)
scores["score_per_10k"] = (scores["newspaper_count"]/scores["population"]*10_000).round(3)
scores["is_news_desert"] = scores["newspaper_count"] == 0

# --- Study area: Region Syddanmark (23 municipalities incl. Horsens code 615) ---
syd = scores[scores["region"] == "Syddanmark"].reset_index(drop=True)

# Explicit guard: all downstream analysis must use  (n=23), not  (n=98).
# If this assertion fails, the filter or dk_municipalities.csv has been changed.
assert len(syd) == 23, f"Studieområdet skal være 23 kommuner — fik {len(syd)}"
assert "Horsens" in syd["kommune_name"].values, "Horsens (kode 615) mangler i studieområdet"
print(f"✓ Filter-tjek: {len(syd)} kommuner, Horsens inkluderet")
print(f"Studieområde n = {len(syd)}")
print(f"Mean score   = {syd['score_per_10k'].mean():.3f}   (rapport: 0.621)")
print(f"Median score = {syd['score_per_10k'].median():.3f}   (rapport: 0.573)")
print(f"Min / Max    = {syd['score_per_10k'].min():.3f} / {syd['score_per_10k'].max():.3f}")
print(f"News deserts (count==0): {syd['is_news_desert'].sum()}   (rapport: 1, Fanø)")

In [ ]:
print("── Top 5 (højeste per-capita score) ──")
print(syd.nlargest(5, 'score_per_10k')[
    ['kommune_name','population','newspaper_count','score_per_10k']]
    .to_string(index=False))

print("\n── Bund 5 (laveste per-capita score) ──")
print(syd.nsmallest(5, 'score_per_10k')[
    ['kommune_name','population','newspaper_count','score_per_10k']]
    .to_string(index=False))

syd.to_csv("syddanmark_scores.csv", index=False)
print("\n→ Gemt: syddanmark_scores.csv")

## 5. Spatial join and choropleth maps

Merges the coverage scores with the GeoJSON polygon layer to enable mapping.

In [ ]:
gdf = gpd.read_file(GEOJSON_PATH)
gdf["kode"] = gdf["kode"].astype(int)
gdf_syd = gdf.merge(syd, left_on="kode", right_on="kommune_kode", how="inner").reset_index(drop=True)
print(f"Geometrier i studieområdet: {len(gdf_syd)}")
print(f"CRS: {gdf_syd.crs}")

### Figure 1 — newspaper count and per-capita score (matches PDF Figure 1)

Two side-by-side panels:
- **Left:** raw newspaper count (`newspaper_count`) — how many unique titles cover each municipality
- **Right:** per-capita score (`score_per_10k`) — newspapers per 10,000 residents

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Left panel: raw newspaper count
gdf_syd.plot(column="newspaper_count", ax=axes[0], legend=True, cmap="YlOrRd",
             edgecolor="white", linewidth=0.4,
             legend_kwds={"label":"Antal lokale aviser", "shrink":0.6})
axes[0].set_title("Antal lokale aviser per kommune", fontweight="bold")
axes[0].axis("off")

# Right panel: per-capita score
gdf_syd.plot(column="score_per_10k", ax=axes[1], legend=True, cmap="RdYlGn",
             edgecolor="white", linewidth=0.4,
             legend_kwds={"label":"Aviser per 10.000 indb.", "shrink":0.6})
axes[1].set_title("Per-capita nyhedsdækning", fontweight="bold")
axes[1].axis("off")

plt.suptitle("Region Syddanmark — nyhedsdækning (Figur 1)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("choropleth_syddanmark.png", dpi=200, bbox_inches="tight")
plt.show()
print("→ Gemt: choropleth_syddanmark.png")

### Figure 2 — binary news desert classification

Strict definition: `newspaper_count == 0`. Within the study area only Fanø meets this criterion.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
colors = gdf_syd['is_news_desert'].map({True:'#d7191c', False:'#2c7bb6'})
gdf_syd.plot(ax=ax, color=colors, edgecolor='white', linewidth=0.4)
ax.legend(handles=[
    mpatches.Patch(facecolor="#d7191c", label="News desert (0 papers)"),
    mpatches.Patch(facecolor="#2c7bb6", label="Has local coverage")
], loc='lower left', fontsize=10)
ax.set_title("Binær news desert-klassifikation — Region Syddanmark (Figur 2)",
             fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("news_deserts_binary_syd.png", dpi=200, bbox_inches="tight")
plt.show()
print("→ Gemt: news_deserts_binary_syd.png")

## 6. Distribution plots

Histograms of raw newspaper count and per-capita score across the 23 municipalities.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(syd['newspaper_count'],
             bins=range(0, syd['newspaper_count'].max()+2),
             color='#2c7bb6', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Antal aviser'); axes[0].set_ylabel('Antal kommuner')
axes[0].set_title('Fordeling: aviser per kommune (n = 23)')

axes[1].hist(syd['score_per_10k'], bins=12, color='#d7191c', edgecolor='white', alpha=0.85)
axes[1].axvline(syd['score_per_10k'].mean(), color='black', linestyle='--',
                label=f"Gns: {syd['score_per_10k'].mean():.3f}")
axes[1].axvline(syd['score_per_10k'].median(), color='grey', linestyle=':',
                label=f"Median: {syd['score_per_10k'].median():.3f}")
axes[1].legend()
axes[1].set_xlabel('Aviser per 10.000 indb.'); axes[1].set_ylabel('Antal kommuner')
axes[1].set_title('Fordeling: per-capita score')
plt.tight_layout(); plt.savefig("distribution_syd.png", dpi=200, bbox_inches="tight"); plt.show()

## 7. Spatial autocorrelation — Global Moran's I

The report (§4.3) uses two weight specifications:

| Weights | Moran's I | p (999 perm.) |
|---|---|---|
| Queen | −0.012 | 0.394 |
| KNN-4 | +0.098 | 0.123 |

**Note on KNN-4:** distances must be computed in a projected CRS (EPSG:25832, UTM zone 32N — the standard for Denmark). Using `KNN.from_dataframe(gdf, k=4)` directly on lon/lat geometry mixes degrees of longitude and latitude (1° lon ≠ 1° lat at 55°N), which changes the neighbour list and therefore the value of I.

In [ ]:
y = gdf_syd["score_per_10k"].values

# --- Queen contiguity weights ---
w_queen = Queen.from_dataframe(gdf_syd, use_index=False)
w_queen.transform = "R"
np.random.seed(RANDOM_SEED)
mi_q = Moran(y, w_queen, permutations=999)
print(f"Queen Moran's I = {mi_q.I:+.4f},  p = {mi_q.p_sim:.4f}   (rapport: -0.012, 0.394)")
print(f"  Forventet I  = {mi_q.EI:+.4f}")
print(f"  ⚠ Queen-vægte advarer om islands (Langeland, Ærø, Fanø, Horsens)")

In [ ]:
# --- KNN-4 from projected centroids (EPSG:25832) ---
g_proj = gdf_syd.to_crs(25832)
centroids = np.array([(p.x, p.y) for p in g_proj.geometry.centroid])

w_knn = KNN.from_array(centroids, k=4)
w_knn.transform = "R"

np.random.seed(RANDOM_SEED)
mi_k = Moran(y, w_knn, permutations=999)
print(f"KNN-4 Moran's I = {mi_k.I:+.4f},  p = {mi_k.p_sim:.4f}   (rapport: +0.098, 0.123)")
print(f"  Forventet I  = {mi_k.EI:+.4f}")
print(f"  Hver kommune har præcis 4 naboer — ingen islands.")

## 8. Moran scatterplot (KNN-4)

Plots standardised per-capita score against its spatially lagged equivalent. The slope of the regression line equals Moran's I.

In [ ]:
y_std = (y - y.mean()) / y.std()
lag_y_std = w_knn.sparse.dot(y_std)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_std, lag_y_std, s=45, color='#2c7bb6', alpha=0.75, edgecolor='white')
ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)

z = np.polyfit(y_std, lag_y_std, 1)
xs = np.linspace(y_std.min(), y_std.max(), 100)
ax.plot(xs, np.poly1d(z)(xs), color='#d7191c', lw=1.5)

for i, name in enumerate(gdf_syd['kommune_name']):
    if abs(y_std[i]) > 1.2 or abs(lag_y_std[i]) > 1.0:
        ax.annotate(name, (y_std[i], lag_y_std[i]), fontsize=8,
                    xytext=(4,3), textcoords='offset points')

ax.set_xlabel("Standardiseret score per 10k")
ax.set_ylabel("Spatial lag (KNN-4)")
ax.set_title(f"Moran scatterplot — KNN-4   (I = {mi_k.I:+.3f}, p = {mi_k.p_sim:.3f})",
             fontweight='bold')
plt.tight_layout(); plt.savefig("moran_scatter_syd.png", dpi=200, bbox_inches="tight"); plt.show()

## 9. LISA — Local Moran's I (KNN-4)

Report §4.3: of the 23 units, only Svendborg is significant at p < 0.05 (LISA p ≈ 0.007), classified as a Low-High outlier. Note that p-values are permutation-based, so small variation around `RANDOM_SEED` is expected.

In [ ]:
np.random.seed(RANDOM_SEED)
lisa = Moran_Local(y, w_knn, permutations=999)
gdf_syd["lisa_q"] = lisa.q
gdf_syd["lisa_p"] = lisa.p_sim
gdf_syd["lisa_sig"] = gdf_syd["lisa_p"] < 0.05
gdf_syd["lisa_cluster"] = gdf_syd.apply(
    lambda r: r["lisa_q"] if r["lisa_sig"] else 0, axis=1)

quad_label = {1:"High-High", 2:"Low-High", 3:"Low-Low", 4:"High-Low", 0:"Ikke sig."}
print("── LISA: signifikante kommuner (p < 0.05) ──")
sig = gdf_syd[gdf_syd["lisa_sig"]][["kommune_name","score_per_10k","lisa_q","lisa_p"]]
sig["kvadrant"] = sig["lisa_q"].map(quad_label)
print(sig.to_string(index=False))

print("\n── Tæt-på-signifikante (0.05 ≤ p < 0.11) ──")
near = gdf_syd[(gdf_syd["lisa_p"]>=0.05) & (gdf_syd["lisa_p"]<0.11)]
print(near[["kommune_name","score_per_10k","lisa_p"]].to_string(index=False))

In [ ]:
cluster_colors = {1:"#d7191c", 2:"#abd9e9", 3:"#2c7bb6", 4:"#fdae61", 0:"#eeeeee"}

fig, ax = plt.subplots(figsize=(9, 9))
for q_val, color in cluster_colors.items():
    sub = gdf_syd[gdf_syd["lisa_cluster"] == q_val]
    if len(sub) > 0:
        sub.plot(ax=ax, color=color, edgecolor='white', linewidth=0.4,
                 label=f"{quad_label[q_val]} (n={len(sub)})")

ax.legend(loc='lower left', fontsize=9)
ax.set_title("LISA klyngekort (KNN-4, p < 0.05) — Region Syddanmark", fontweight='bold')
ax.axis("off")
plt.tight_layout(); plt.savefig("lisa_clusters_syd.png", dpi=200, bbox_inches="tight"); plt.show()

## 10. Socioeconomic correlations (Spearman + Pearson)

Expected Spearman correlations (report §4.5, Table 1):

| Variable | Spearman ρ | p | Pearson r | p |
|---|---|---|---|---|
| Mean disposable income | −0.577 | 0.004 | −0.478 | 0.021 |
| Higher-education share | −0.630 | 0.001 | −0.572 | 0.004 |
| Voter turnout (KV21) | +0.265 | 0.222 | +0.270 | 0.212 |

Also report §4.4: Spearman ρ(population, score) = −0.558, p = 0.006.

In [ ]:
socio = pd.read_csv(SOCIO_CSV)
m = syd.merge(socio, on="kommune_kode", how="left")
print(f"Merged with socio: n = {len(m)}")
print(f"Manglende værdier: {m[['avg_income','pct_higher_education','voter_turnout_pct']].isna().sum().to_dict()}")

In [ ]:
rows = []
for var, label in [("avg_income",            "Mean disposable income (kr)"),
                   ("pct_higher_education",  "Higher-education share (%)"),
                   ("voter_turnout_pct",     "Voter turnout, KV21 (%)")]:
    valid = m[["score_per_10k", var]].dropna()
    rho, p_s = spearmanr(valid["score_per_10k"], valid[var])
    r,   p_p = pearsonr (valid["score_per_10k"], valid[var])
    rows.append({"Variable": label, "Spearman ρ": round(rho,3), "p (S)": round(p_s,3),
                 "Pearson r":  round(r,3),  "p (P)": round(p_p,3),
                 "n": len(valid)})

# Also check population correlation
rho_pop, p_pop = spearmanr(m["score_per_10k"], m["population"])
r_pop,   p_pop_pearson = pearsonr(m["score_per_10k"], m["population"])
rows.append({"Variable":"Population (rapport §4.4)",
             "Spearman ρ":round(rho_pop,3), "p (S)":round(p_pop,3),
             "Pearson r":round(r_pop,3),    "p (P)":round(p_pop_pearson,3),
             "n":len(m)})

corr_table = pd.DataFrame(rows)
print(corr_table.to_string(index=False))

## 11. Scatterplots: coverage vs. socioeconomics

In [ ]:
vars_to_plot = [("population",            "Population"),
                ("avg_income",            "Mean disposable income (kr)"),
                ("pct_higher_education",  "Higher-education share (%)"),
                ("voter_turnout_pct",     "Voter turnout KV21 (%)")]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, (var, label) in zip(axes, vars_to_plot):
    valid = m[["score_per_10k", var, "kommune_name"]].dropna()
    ax.scatter(valid[var], valid["score_per_10k"], s=40, alpha=0.7, color='#2c7bb6')
    z = np.polyfit(valid[var], valid["score_per_10k"], 1)
    xs = np.linspace(valid[var].min(), valid[var].max(), 100)
    ax.plot(xs, np.poly1d(z)(xs), '--', color='#d7191c', lw=1.3)

    rho, p_s = spearmanr(valid["score_per_10k"], valid[var])
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel("Aviser per 10k", fontsize=10)
    ax.set_title(f"Spearman ρ = {rho:+.3f}, p = {p_s:.3f}", fontsize=10)
plt.tight_layout(); plt.savefig("correlation_scatter_syd.png", dpi=200, bbox_inches="tight"); plt.show()

## 12. Robustness check: exclude Horsens (n = 22)

Report §4.5 notes that the correlations are qualitatively unchanged when Horsens is excluded (n = 22, the official Region Syddanmark municipality count):
ρ_income = −0.574 (p = 0.005), ρ_edu = −0.642 (p = 0.001), ρ_turnout = +0.299 (p = 0.176).

In [ ]:
m22 = m[m["kommune_name"] != "Horsens"]
print(f"n = {len(m22)} (Horsens ekskluderet)")
for var, label in [("avg_income",            "Income"),
                   ("pct_higher_education",  "Edu"),
                   ("voter_turnout_pct",     "Turnout"),
                   ("population",            "Population")]:
    valid = m22[["score_per_10k", var]].dropna()
    rho, p_s = spearmanr(valid["score_per_10k"], valid[var])
    print(f"  {label:<10s} Spearman ρ = {rho:+.3f}, p = {p_s:.3f}")

## 13. Summary check against report

If the numbers above match the report within ±0.01 (Moran's I) and ±0.05 (permutation-based p-values), this notebook fully reproduces §4–§4.5 of `news_deserts_corrected.tex`.

To do:
- [ ] Confirm that `socioeconomic_data.csv` matches the exact table versions used in the report (INDKP101 year, HFUDD10 year, KV21 source)
- [ ] If KNN-4 p-values differ slightly, try varying `RANDOM_SEED` — the global I is deterministic but p_sim is permutation-based